# 1/2：Kaggle内でCASMIの依存ファイルとcatalogを準備

PCでのZIP復元・大きいファイルのアップロードは不要です。準備をKaggle内で1回行い、
保存した出力をもう一方の提出Notebookへ入力として追加します。

1. このNotebookをインポートし、**private・CPU・Internet ON** にします
2. 競技入力を追加します。準備に必要なのは `train.parquet` だけです
3. **Save Version → Save & Run All** を選び、保存用の実行を1回開始します。
   先に手動で全セルを実行する必要はありません
4. 固定wheel（約131MB）を公式配布元から取得してSHA256を照合します。
   catalogがなければtrainから作成し、既存catalogが入力にあれば検証して再利用します
5. 実行完了まで待ち、成功したversionの出力を確認します。
   Quick Saveだけでは実行済みの出力は作れません。画面の表記は変わる場合があります
6. `casmi_kaggle_submit_from_output_py313_20261005.ipynb` に、この準備Notebookの保存済みOutputと
   競技入力を追加します。提出側は **Internet OFF・CPU・fresh kernel** で実行します

出力は `/kaggle/working/casmi_prepared_assets_v1/` です。中の `.whl` と `.sqlite` が
提出Notebookの入力欄に見える状態にします。ZIPやpartファイルを操作する必要はありません。
完成時に `asset_manifest.json` が作られます。
競技データを含むためNotebookとOutputはprivateを維持します。

初回catalog作成は重い処理です。2CPU workerで進捗を表示します。
Kaggle実機での全件所要時間は未計測です。過去の完成catalogは復旧したものを検証しています。
この版で全件cold buildが時間内に完走する保証はありません。
既存catalogを利用すれば構築を省けます。途中停止時は完成と扱わず、エラーとcheckpointを残します。
この準備Notebookではtestを読まず、予測・評価・競技提出も実行しません。


In [1]:
# 通常はNoneのままです。同名入力が複数の場合だけ実際のパスを指定します。
TRAIN_FILE = None
CATALOG_FILE = None  # 既存の完成catalogがあれば任意。なければtrainから構築します。


## 入力確認と準備の実行
大きいソースセルは編集不要です。

In [2]:
import json
import os
import platform
import subprocess
import sys
import tempfile
from pathlib import Path

if sys.implementation.name != "cpython" or sys.version_info[:2] != (3, 13):
    raise RuntimeError("CPython 3.13用です。表示されたruntimeに対応する版を使ってください。")
print("Python:", platform.python_version())
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
input_root = Path(os.environ.get("CASMI_INPUT_ROOT", "/kaggle/input"))
work_root = Path(os.environ.get("CASMI_WORK_ROOT", "/kaggle/working"))
work_root.mkdir(parents=True, exist_ok=True)
train_override = os.environ.get("CASMI_TRAIN_PATH") or TRAIN_FILE
train_paths = ([Path(train_override)] if train_override
               else sorted(input_root.rglob("train.parquet")))
if len(train_paths) != 1 or not train_paths[0].is_file():
    raise RuntimeError("競技train.parquetを1つ追加します。重複時はTRAIN_FILEを指定します。")
catalog_override = os.environ.get("CASMI_CATALOG_PATH") or CATALOG_FILE
catalog_paths = ([Path(catalog_override)] if catalog_override
                 else sorted(input_root.rglob("catalog.sqlite")))
if len(catalog_paths) > 1:
    raise RuntimeError("catalogが複数あります。CATALOG_FILEで再利用する1つを指定してください。")
source_dir = Path(tempfile.mkdtemp(prefix="casmi-preparation-source-"))

EMBEDDED_SOURCE = {'build_reference_catalog_v2.py': '"""Bounded, checkpoint-recoverable private CASMI catalog builder.\n\nUses at most four small in-flight chunks and reports process failures/timeouts\nexplicitly. Normalization semantics are unchanged from the original builder.\nExisting files/directories are never overwritten; --resume-from is read-only.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport concurrent.futures as cf\nimport hashlib\nimport json\nimport multiprocessing as mp\nimport resource\nimport sqlite3\nimport time\nfrom pathlib import Path\n\nimport pyarrow.parquet as pq\nfrom casmi_chemistry import (\n    normalize_record,\n    runtime_versions,\n    sha256_file,\n    structure_mass,\n    write_json,\n)\n\nNORMALIZATION_CONTRACT = "RDKit-default-TautomerEnumerator-Canonicalize-InChIKey14-v1"\n\n\ndef normalize_chunk(rows):\n    return [(row, normalize_record(row)) for row in rows]\n\n\ndef raw_digest(db):\n    digest = hashlib.sha256()\n    for row in db.execute(\n        "SELECT smiles,provided,formula FROM raw ORDER BY smiles,provided,formula"\n    ):\n        digest.update(json.dumps(row, separators=(",", ":")).encode())\n        digest.update(b"\\n")\n    return digest.hexdigest()\n\n\ndef terminate_executor(executor):\n    # Python 3.12 lacks the public terminate_workers API (added in 3.14).\n    # These are only child processes owned by this executor, never other jobs.\n    for process in list((executor._processes or {}).values()):\n        process.terminate()\n    executor.shutdown(wait=False, cancel_futures=True)\n\n\ndef build(\n    train: Path,\n    output: Path,\n    workers: int = 2,\n    resume_from: Path | None = None,\n    chunk_timeout_seconds: float = 600,\n    max_rows: int | None = None,\n    legacy_contract: Path | None = None,\n) -> dict:\n    output.mkdir(parents=True, exist_ok=False)\n    start = time.monotonic()\n    train_hash = sha256_file(train)\n    versions = runtime_versions()\n    write_json(\n        output / "build_contract.json",\n        {\n            "train_sha256": train_hash,\n            "versions": versions,\n            "normalization_contract": NORMALIZATION_CONTRACT,\n            "chunk_timeout_seconds": chunk_timeout_seconds,\n            "chunk_rows": 16,\n            "on_timeout_or_worker_failure": "fail visibly; retain checkpoint; never guess a key",\n        },\n    )\n    db = sqlite3.connect(output / "catalog.sqlite")\n    db.execute("PRAGMA cache_size=-16000")\n    db.execute(\n        "CREATE TABLE raw(smiles TEXT,provided TEXT,formula TEXT,"\n        "PRIMARY KEY(smiles,provided,formula)) WITHOUT ROWID"\n    )\n    db.execute(\n        "CREATE TABLE structures(smiles TEXT,provided TEXT,formula TEXT,proxy TEXT,"\n        "canonical TEXT,mass REAL,PRIMARY KEY(smiles,provided,formula)) WITHOUT ROWID"\n    )\n    db.execute("CREATE TABLE failures(smiles TEXT,provided TEXT,formula TEXT,reason TEXT)")\n    db.execute("CREATE TABLE provenance(key TEXT PRIMARY KEY,value TEXT)")\n    db.executemany(\n        "INSERT INTO provenance VALUES (?,?)",\n        [\n            ("train_sha256", train_hash),\n            ("rdkit_version", versions["rdkit"]),\n            ("normalization_contract", NORMALIZATION_CONTRACT),\n        ],\n    )\n    columns = ["normalized_smiles", "inchikey14", "molecular_formula"]\n    count = 0\n    pf = pq.ParquetFile(train, memory_map=True, pre_buffer=False)\n    for batch in pf.iter_batches(batch_size=4096, columns=columns, use_threads=False):\n        rows = list(zip(*(batch.column(c).to_pylist() for c in columns), strict=True))\n        if max_rows is not None:\n            rows = rows[: max(0, max_rows - count)]\n        for smiles, provided, formula in rows:\n            if smiles and provided:\n                db.execute(\n                    "INSERT OR IGNORE INTO raw VALUES (?,?,?)",\n                    (smiles, provided, formula if formula is not None else ""),\n                )\n        count += len(rows)\n        if max_rows is not None and count >= max_rows:\n            break\n    db.commit()\n    unique = db.execute("SELECT COUNT(*) FROM raw").fetchone()[0]\n    reused = mass_changed = 0\n    if resume_from is not None:\n        source_contract_path = resume_from.with_name("build_contract.json")\n        if source_contract_path.is_file():\n            source_contract = json.loads(source_contract_path.read_text())\n        elif legacy_contract is not None:\n            source_contract = json.loads(legacy_contract.read_text())\n            if source_contract.get("catalog_sha256") != sha256_file(resume_from):\n                raise ValueError("Legacy normalization assertion checksum mismatch")\n        else:\n            raise ValueError(\n                "Resume requires matching build_contract.json or an explicit "\n                "--legacy-contract assertion for a known historical checkpoint"\n            )\n        if (\n            source_contract.get("versions", {}).get("rdkit") != versions["rdkit"]\n            or source_contract.get("normalization_contract") != NORMALIZATION_CONTRACT\n            or source_contract.get("train_sha256") != train_hash\n        ):\n            raise ValueError("Resume RDKit/normalization/train contract mismatch")\n        source = sqlite3.connect("file:" + str(resume_from.resolve()) + "?mode=ro", uri=True)\n        # Bind an older partial cache to the complete same structural input set.\n        # Reuse is only for catalogs produced by this project\'s unchanged normalizer.\n        if raw_digest(source) != raw_digest(db):\n            raise ValueError("Resume catalog structural input-set mismatch")\n        for row in source.execute(\n            "SELECT smiles,provided,formula,proxy,canonical,mass FROM structures"\n        ):\n            mass = structure_mass(row[0], row[2])\n            mass_changed += mass != row[5]\n            db.execute("INSERT INTO structures VALUES (?,?,?,?,?,?)", (*row[:5], mass))\n            reused += 1\n            if reused % 10000 == 0:\n                db.commit()\n                print(f"recovered {reused} cached identities", flush=True)\n        source.close()\n        db.commit()\n    print(f"catalog input_rows={count} variants={unique} reused={reused}", flush=True)\n    cursor = db.execute(\n        "SELECT r.smiles,r.provided,r.formula FROM raw r "\n        "LEFT JOIN structures s USING(smiles,provided,formula) "\n        "WHERE s.smiles IS NULL ORDER BY r.smiles,r.provided,r.formula"\n    )\n    # Materialize only row IDs/strings here; no spectral arrays or RDKit objects.\n    # Main-thread fetch avoids SQLite cross-thread iterator access in task feeder.\n    pending_rows = cursor.fetchall()\n    workers = min(2, max(1, workers))\n    executor = cf.ProcessPoolExecutor(max_workers=workers, mp_context=mp.get_context("spawn"))\n    pending = {}\n    position = processed = invalid = 0\n    last_log = time.monotonic()\n    failed = False\n    try:\n        while position < len(pending_rows) or pending:\n            while position < len(pending_rows) and len(pending) < workers * 2:\n                rows = pending_rows[position : position + 16]\n                future = executor.submit(normalize_chunk, rows)\n                pending[future] = (time.monotonic(), position, rows)\n                position += len(rows)\n            finished, _ = cf.wait(pending, timeout=1.0, return_when=cf.FIRST_COMPLETED)\n            for future in finished:\n                _, _, rows = pending.pop(future)\n                for raw, result in future.result():\n                    processed += 1\n                    if result is None:\n                        invalid += 1\n                        db.execute(\n                            "INSERT INTO failures VALUES (?,?,?,?)",\n                            (*raw, "invalid RDKit normalization"),\n                        )\n                    else:\n                        db.execute("INSERT INTO structures VALUES (?,?,?,?,?,?)", result)\n                db.commit()\n            now = time.monotonic()\n            expired = [\n                (pos, rows)\n                for began, pos, rows in pending.values()\n                if now - began > chunk_timeout_seconds\n            ]\n            if expired:\n                write_json(output / "timed_out_chunks_PRIVATE.json", expired)\n                raise TimeoutError(\n                    "Normalization chunk deadline exceeded; private checkpoint saved"\n                )\n            if now - last_log >= 15 or not pending:\n                print(\n                    f"normalized={reused + processed}/{unique} pending_chunks={len(pending)} "\n                    f"seconds={now - start:.1f}",\n                    flush=True,\n                )\n                last_log = now\n    except BaseException as error:\n        failed = True\n        db.commit()\n        write_json(\n            output / "build_failure.json",\n            {\n                "error_type": type(error).__name__,\n                "message": str(error),\n                "processed_new": processed,\n                "reused": reused,\n                "complete": False,\n                "wall_seconds": time.monotonic() - start,\n            },\n        )\n        terminate_executor(executor)\n        raise\n    finally:\n        if not failed:\n            executor.shutdown(wait=True)\n    db.execute("CREATE INDEX structures_provided ON structures(provided)")\n    db.execute("CREATE INDEX structures_proxy ON structures(proxy)")\n    db.execute("CREATE INDEX structures_mass ON structures(mass)")\n    db.commit()\n    valid = db.execute("SELECT COUNT(*) FROM structures").fetchone()[0]\n    proxies = db.execute("SELECT COUNT(DISTINCT proxy) FROM structures").fetchone()[0]\n    unsupported = db.execute("SELECT COUNT(*) FROM structures WHERE mass IS NULL").fetchone()[0]\n    db.close()\n    result = {\n        "kind": "private train-derived reference catalog",\n        "versions": versions,\n        "normalization_contract": NORMALIZATION_CONTRACT,\n        "train_sha256": train_hash,\n        "input_rows": count,\n        "unique_variants": unique,\n        "valid_variants": valid,\n        "invalid_variants": invalid,\n        "proxy_structures": proxies,\n        "unsupported_mass_variants": unsupported,\n        "reused_normalized_variants": reused,\n        "reused_mass_changed_variants": mass_changed,\n        "resumed_from_sha256": sha256_file(resume_from) if resume_from else None,\n        "limited_smoke": max_rows is not None,\n        "workers": workers,\n        "wall_seconds": time.monotonic() - start,\n        "parent_peak_rss_kib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,\n        "worker_peak_rss_kib": resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss,\n        "catalog_sha256": sha256_file(output / "catalog.sqlite"),\n    }\n    write_json(output / "manifest.json", result)\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--train", type=Path, required=True)\n    parser.add_argument("--output", type=Path, required=True)\n    parser.add_argument("--resume-from", type=Path)\n    parser.add_argument("--workers", type=int, default=2)\n    parser.add_argument("--chunk-timeout-seconds", type=float, default=600)\n    parser.add_argument("--max-rows", type=int)\n    parser.add_argument("--legacy-contract", type=Path)\n    args = parser.parse_args()\n    print(\n        build(\n            args.train,\n            args.output,\n            args.workers,\n            args.resume_from,\n            args.chunk_timeout_seconds,\n            args.max_rows,\n            args.legacy_contract,\n        ),\n        flush=True,\n    )\n\n\nif __name__ == "__main__":\n    main()\n', 'casmi_chemistry.py': '"""CASMI CPU baseline chemical contracts; local metric proxy, NOT official scorer."""\n\nfrom __future__ import annotations\n\nimport functools\nimport hashlib\nimport json\nimport math\nimport re\nfrom collections import Counter\nfrom pathlib import Path\n\nimport numpy as np\nfrom rdkit import Chem, RDLogger, rdBase\nfrom rdkit.Chem import rdMolDescriptors\nfrom rdkit.Chem.MolStandardize import rdMolStandardize\n\nRDLogger.DisableLog("rdApp.*")\nELECTRON_MASS = 0.000548579909065\nPT = Chem.GetPeriodicTable()\n\n\ndef sha256_file(path: Path) -> str:\n    h = hashlib.sha256()\n    with path.open("rb") as stream:\n        while block := stream.read(8 * 1024 * 1024):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write_json(path: Path, value: object) -> None:\n    with path.open("x", encoding="utf-8") as stream:\n        json.dump(value, stream, indent=2, sort_keys=True, allow_nan=False)\n        stream.write("\\n")\n\n\n@functools.lru_cache(maxsize=100000)\ndef atom_mass(formula: str | None) -> float | None:\n    if not isinstance(formula, str) or not formula:\n        return None\n    terms = list(re.finditer(r"([A-Z][a-z]?)([0-9]*)", formula))\n    if "".join(x.group() for x in terms) != formula:\n        return None\n    counts: Counter[str] = Counter()\n    try:\n        for term in terms:\n            element, count = term.groups()\n            count = int(count or 1)\n            if PT.GetAtomicNumber(element) <= 0 or count <= 0:\n                return None\n            counts[element] += count\n        return sum(PT.GetMostCommonIsotopeMass(e) * n for e, n in counts.items())\n    except (RuntimeError, ValueError):\n        return None\n\n\n@functools.lru_cache(maxsize=1000)\ndef adduct_parts(adduct: str | None) -> tuple[int, int, float] | None:\n    if not isinstance(adduct, str):\n        return None\n    match = re.fullmatch(r"\\[(\\d*)M((?:[+-][A-Za-z0-9]+)*)\\](\\d*)([+-])", adduct)\n    if match is None:\n        return None\n    mult = int(match.group(1) or 1)\n    charge = int(match.group(3) or 1) * (1 if match.group(4) == "+" else -1)\n    if mult <= 0 or charge == 0:\n        return None\n    delta = 0.0\n    for sign, term in re.findall(r"([+-])([A-Za-z0-9]+)", match.group(2)):\n        part = re.fullmatch(r"(\\d*)([A-Z].*)", term)\n        if part is None:\n            return None\n        mass = atom_mass(part.group(2))\n        count = int(part.group(1) or 1)\n        if mass is None or count <= 0:\n            return None\n        delta += (1 if sign == "+" else -1) * count * mass\n    return mult, charge, delta\n\n\ndef neutral_mass(precursor: float | None, adduct: str | None) -> float | None:\n    parts = adduct_parts(adduct)\n    if parts is None or precursor is None or not math.isfinite(precursor) or precursor <= 0:\n        return None\n    mult, charge, delta = parts\n    value = (precursor * abs(charge) - delta + charge * ELECTRON_MASS) / mult\n    return value if value > 0 else None\n\n\n@functools.lru_cache(maxsize=100000)\ndef normalize_smiles(smiles: str | None) -> tuple[str, str] | None:\n    """Default RDKit tautomer canonicalization then InChIKey14, without salt cleanup.\n\n    This is an explicit *local proxy*. The official scorer\'s complete source has\n    not been supplied; version equality does not establish scorer parity.\n    """\n    if not isinstance(smiles, str) or not smiles.strip():\n        return None\n    mol = Chem.MolFromSmiles(smiles)\n    if mol is None or mol.GetNumAtoms() == 0:\n        return None\n    try:\n        canonical = rdMolStandardize.TautomerEnumerator().Canonicalize(mol)\n        key = Chem.MolToInchiKey(canonical)\n        if not key or len(key) < 14:\n            return None\n        return key[:14], Chem.MolToSmiles(canonical, isomericSmiles=True)\n    except (RuntimeError, ValueError):\n        return None\n\n\ndef structure_mass(smiles: str, formula: str | None) -> float | None:\n    """Atomic mass sum; terminal formula charge is not elemental composition.\n\n    Charged formula suffixes (+, -, +2, -2) are removed explicitly. Isotopic or\n    nonstandard formula grammar falls back to the actual candidate structure.\n    No query formula or inferred neutralization is used.\n    """\n    mol = Chem.MolFromSmiles(smiles)\n    if mol is None or any(atom.GetAtomicNum() == 0 for atom in mol.GetAtoms()):\n        return None\n    # Supplied formulas can omit isotope labels; the structure is authoritative here.\n    isotopic = any(atom.GetIsotope() for atom in mol.GetAtoms())\n    uncharged = re.sub(r"[+-]\\d*$", "", formula) if isinstance(formula, str) else None\n    mass = None if isotopic else atom_mass(uncharged)\n    if mass is not None:\n        return mass\n    mass = rdMolDescriptors.CalcExactMolWt(mol) + Chem.GetFormalCharge(mol) * ELECTRON_MASS\n    return float(mass) if mass > 0 and math.isfinite(mass) else None\n\n\ndef normalize_record(row: tuple[str, str, str | None]) -> tuple | None:\n    smiles, provided, formula = row\n    norm = normalize_smiles(smiles)\n    if norm is None:\n        return None\n    mass = structure_mass(smiles, formula)\n    return smiles, provided, formula, norm[0], norm[1], mass\n\n\ndef clean_peaks(mzs, intensities) -> tuple[np.ndarray, np.ndarray]:\n    x = np.asarray(mzs if mzs is not None else [], dtype=np.float64)\n    y = np.asarray(intensities if intensities is not None else [], dtype=np.float64)\n    if x.ndim != 1 or y.ndim != 1 or len(x) != len(y):\n        raise ValueError("m/z and intensity arrays must be equal-length vectors")\n    ok = np.isfinite(x) & np.isfinite(y) & (x > 0) & (y > 0)\n    x, y = x[ok], y[ok]\n    order = np.argsort(x, kind="stable")\n    x, y = x[order], y[order]\n    if len(x):\n        unique, inverse = np.unique(x, return_inverse=True)\n        y = np.bincount(inverse, weights=y)\n        x = unique\n        y /= y.sum()\n    return x, y\n\n\ndef peak_hash(mzs, intensities) -> str:\n    """Canonical peak array identity, ignoring row IDs and acquisition metadata."""\n    x, y = clean_peaks(mzs, intensities)\n    h = hashlib.sha256()\n    h.update(x.astype("<f8").tobytes())\n    h.update(y.astype("<f8").tobytes())\n    return h.hexdigest()\n\n\ndef binned_peaks(mzs, intensities, bin_width: float) -> tuple[np.ndarray, np.ndarray]:\n    """Variable-length fixed-bin sqrt-intensity cosine; no top-N truncation."""\n    if bin_width <= 0:\n        raise ValueError("bin width must be positive")\n    x, y = clean_peaks(mzs, intensities)\n    bins, inverse = np.unique(np.floor(x / bin_width).astype(np.int64), return_inverse=True)\n    weights = np.sqrt(np.bincount(inverse, weights=y))\n    norm = np.linalg.norm(weights)\n    return bins, weights / norm if norm else weights\n\n\ndef sparse_cosine(left: tuple, right: tuple) -> float:\n    a, av = left\n    b, bv = right\n    if not len(a) or not len(b):\n        return 0.0\n    if len(a) > len(b):\n        a, av, b, bv = b, bv, a, av\n    pos = np.searchsorted(b, a)\n    ok = pos < len(b)\n    ids = np.flatnonzero(ok)\n    ids = ids[b[pos[ids]] == a[ids]]\n    return float(np.clip(np.dot(av[ids], bv[pos[ids]]), 0, 1))\n\n\ndef proxy_reciprocal_rank(target: str, candidates: list[str], limit: int = 25) -> float:\n    truth = normalize_smiles(target)\n    if truth is None:\n        raise ValueError("Invalid target structure")\n    # Raw candidate positions count; invalid or repeated entries do not move later ranks up.\n    for rank, candidate in enumerate(candidates[:limit], 1):\n        norm = normalize_smiles(candidate)\n        if norm is not None and norm[0] == truth[0]:\n            return 1.0 / rank\n    return 0.0\n\n\ndef runtime_versions() -> dict:\n    import importlib.metadata\n    import platform\n\n    return {\n        "python": platform.python_version(),\n        "rdkit": rdBase.rdkitVersion,\n        "numpy": importlib.metadata.version("numpy"),\n        "pyarrow": importlib.metadata.version("pyarrow"),\n        "official_metric_parity": False,\n    }\n', 'casmi_config_v1.json': '{\n  "name": "casmi_cpu_v1_frozen_20261005",\n  "description": "Training-free diagnostic baseline; fixed before validation; official parity unverified",\n  "seed": "casmi-cpu-baseline-v1-20261005",\n  "mass_ppm": 50.0,\n  "mass_floor_da": 0.02,\n  "mass_score_ppm": 10.0,\n  "mass_score_floor_da": 0.005,\n  "nearest_mass_fallback": 128,\n  "max_candidates": 5000,\n  "bin_width_da": 0.02,\n  "arrow_batch_rows": 256,\n  "rrf_k": 60.0,\n  "mass_weight": 0.25,\n  "spectral_weight": 0.75,\n  "collision_energy_scale_ev": 50.0,\n  "near_duplicate_cosine": 0.995,\n  "near_duplicate_precursor_da": 0.02,\n  "max_predictions": 25,\n  "validation_sources": [\n    "drug_plus",\n    "enveda-180",\n    "gnps",\n    "masaryk",\n    "massbank",\n    "mona",\n    "msdial",\n    "pluskal_ms2",\n    "riken",\n    "spectraverse"\n  ],\n  "validation_per_source": 20,\n  "validation_max_queries": 8,\n  "bootstrap_repetitions": 1000,\n  "model_selection": "No hyperparameter or seed search; all preset variants reported; B2 cosine RRF is the frozen submission default"\n}\n', 'casmi_cpu_baseline.py': '"""Bounded, training-free CASMI retrieval and mass-union baseline.\n\nQuery allowlist contains only test-available columns. IDs are output/grouping\nkeys only. This does not solve candidate-DB-out structures. No network calls.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport collections\nimport csv\nimport json\nimport math\nimport resource\nimport sqlite3\nimport time\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nfrom casmi_chemistry import (\n    adduct_parts,\n    binned_peaks,\n    neutral_mass,\n    normalize_smiles,\n    peak_hash,\n    runtime_versions,\n    sha256_file,\n    sparse_cosine,\n    write_json,\n)\n\nQUERY_COLUMNS = [\n    "molecule_id",\n    "spectrum_id",\n    "ms2_mzs",\n    "ms2_normalized_intensities",\n    "base_peak_intensity",\n    "adduct",\n    "ionization_mode",\n    "instrument_type",\n    "precursor_mz",\n    "collision_energy_ev",\n    "collision_energy_orig",\n    "collision_energy_orig_units",\n]\nREQUIRED_QUERY_COLUMNS = {\n    "molecule_id",\n    "ms2_mzs",\n    "ms2_normalized_intensities",\n    "adduct",\n    "ionization_mode",\n    "precursor_mz",\n}\nTRAIN_COLUMNS = [\n    "normalized_smiles",\n    "inchikey14",\n    "ingest_lib",\n    "ms2_mzs",\n    "ms2_normalized_intensities",\n    "adduct",\n    "ionization_mode",\n    "precursor_mz",\n    "collision_energy_ev",\n]\n\n\n@dataclass\nclass Query:\n    molecule: str\n    peaks: tuple\n    peak_digest: str\n    precursor: float | None\n    adduct: str | None\n    polarity: str | None\n    ce: float | None\n    contradiction: bool\n    condition: tuple\n\n\ndef mean_ce(value) -> float | None:\n    if value is None:\n        return None\n    values = np.asarray(value, dtype=float).reshape(-1)\n    values = values[np.isfinite(values)]\n    return float(values.mean()) if len(values) else None\n\n\ndef query_from_row(row: dict, config: dict) -> Query:\n    # Ignore every non-allowlisted input; labels/source must never affect inference.\n    row = {k: v for k, v in row.items() if k in QUERY_COLUMNS}\n    missing = REQUIRED_QUERY_COLUMNS - row.keys()\n    if missing:\n        raise ValueError(f"Missing required query fields: {sorted(missing)}")\n    if row["molecule_id"] is None:\n        raise ValueError("Null molecule_id")\n    adduct, polarity = row["adduct"], row["ionization_mode"]\n    parts = adduct_parts(adduct)\n    contradiction = bool(\n        parts\n        and polarity in {"positive", "negative"}\n        and ((parts[1] > 0) != (polarity == "positive"))\n    )\n    ce = mean_ce(row.get("collision_energy_ev"))\n    raw_ce = row.get("collision_energy_ev")\n    finite_ce = (\n        tuple(float(v) for v in raw_ce if v is not None and math.isfinite(v))\n        if raw_ce is not None\n        else None\n    )\n    condition = (\n        polarity,\n        adduct,\n        row.get("instrument_type"),\n        finite_ce,\n        row.get("collision_energy_orig_units"),\n    )\n    return Query(\n        str(row["molecule_id"]),\n        binned_peaks(row["ms2_mzs"], row["ms2_normalized_intensities"], config["bin_width_da"]),\n        peak_hash(row["ms2_mzs"], row["ms2_normalized_intensities"]),\n        row["precursor_mz"],\n        adduct,\n        polarity,\n        ce,\n        contradiction,\n        condition,\n    )\n\n\ndef read_queries(path: Path, config: dict) -> tuple[list[Query], list[str], dict]:\n    pf = pq.ParquetFile(path, memory_map=True, pre_buffer=False)\n    missing = REQUIRED_QUERY_COLUMNS - set(pf.schema_arrow.names)\n    if missing:\n        raise ValueError(f"Missing required query columns: {sorted(missing)}")\n    names = [c for c in QUERY_COLUMNS if c in pf.schema_arrow.names]\n    queries, ids, seen, seen_ids = [], [], set(), set()\n    count = 0\n    for batch in pf.iter_batches(\n        batch_size=config["arrow_batch_rows"], columns=names, use_threads=False\n    ):\n        for row in batch.to_pylist():\n            count += 1\n            query = query_from_row(row, config)\n            if query.molecule not in seen_ids:\n                ids.append(query.molecule)\n                seen_ids.add(query.molecule)\n            signature = (query.molecule, query.peak_digest, query.precursor, query.condition)\n            if signature not in seen:\n                queries.append(query)\n                seen.add(signature)\n    if not queries:\n        raise ValueError("Runtime query file is empty")\n    return (\n        queries,\n        ids,\n        {\n            "input_rows": count,\n            "unique_query_rows": len(queries),\n            "duplicate_queries_removed": count - len(queries),\n            "molecules": len(ids),\n            "contradictory_query_rows": sum(q.contradiction for q in queries),\n            "ce_missing_rows": sum(q.ce is None for q in queries),\n            "unknown_adduct_rows": sum(adduct_parts(q.adduct) is None for q in queries),\n        },\n    )\n\n\ndef mass_hypotheses(query: Query) -> list[tuple[float, float, bool]]:\n    """Original interpretation plus limited fallback only for unknown/contradictory rows."""\n    parts = adduct_parts(query.adduct)\n    value = neutral_mass(query.precursor, query.adduct)\n    hypotheses = []\n    if value is not None and parts is not None:\n        hypotheses.append((value, abs(parts[1]) / parts[0], False))\n    if value is None or query.contradiction:\n        modes = (\n            (query.polarity,)\n            if query.polarity in {"positive", "negative"}\n            else ("positive", "negative")\n        )\n        alternatives = []\n        if "positive" in modes:\n            alternatives.extend(["[M+H]+", "[M+Na]+", "[M+NH4]+", "[M+K]+", "[M]+"])\n        if "negative" in modes:\n            alternatives.extend(["[M-H]-", "[M+Cl]-", "[M+CH2O2-H]-", "[M]-"])\n        for adduct in alternatives:\n            mass = neutral_mass(query.precursor, adduct)\n            if mass is not None and not any(abs(mass - h[0]) < 1e-8 for h in hypotheses):\n                hypotheses.append((mass, 1.0, True))\n    return hypotheses\n\n\ndef open_catalog(path: Path) -> sqlite3.Connection:\n    db = sqlite3.connect("file:" + str(path.resolve()) + "?mode=ro", uri=True)\n    db.execute("PRAGMA cache_size=-12000")\n    return db\n\n\ndef make_candidates(\n    queries: list[Query], db: sqlite3.Connection, config: dict, excluded: set[str] | None = None\n) -> tuple[dict, dict, dict]:\n    excluded = excluded or set()\n    groups = collections.defaultdict(list)\n    for query in queries:\n        groups[query.molecule].append(query)\n    pools, canonical, details = {}, {}, {}\n    for molecule, group in groups.items():\n        candidate_masses = collections.defaultdict(set)\n        union, strict_sets, hypotheses = set(), [], set()\n        for query in group:\n            query_union = set()\n            query_hypotheses = mass_hypotheses(query)\n            for center, scale, guessed in query_hypotheses:\n                hypothesis = (center, scale, guessed)\n                hypotheses.add(hypothesis)\n                radius = (\n                    max(float(query.precursor) * config["mass_ppm"] / 1e6, config["mass_floor_da"])\n                    * scale\n                )\n                rows = db.execute(\n                    "SELECT proxy,canonical,mass FROM structures WHERE mass BETWEEN ? AND ?",\n                    (center - radius, center + radius),\n                )\n                current = set()\n                for proxy, smiles, mass in rows:\n                    if proxy in excluded:\n                        continue\n                    current.add(proxy)\n                    canonical[proxy] = min(canonical.get(proxy, smiles), smiles)\n                    candidate_masses[proxy].add(mass)\n                union.update(current)\n                query_union.update(current)\n                # Always add a bounded broad mass-neighbour channel, not only on empty union.\n                for direction, order in ((">=", "ASC"), ("<=", "DESC")):\n                    rows = db.execute(\n                        f"SELECT proxy,canonical,mass FROM structures WHERE mass "\n                        f"{direction} ? ORDER BY mass {order} LIMIT ?",\n                        (center, config["nearest_mass_fallback"]),\n                    )\n                    for proxy, smiles, mass in rows:\n                        if proxy in excluded:\n                            continue\n                        canonical[proxy] = min(canonical.get(proxy, smiles), smiles)\n                        candidate_masses[proxy].add(mass)\n            if query_hypotheses:\n                strict_sets.append(query_union)\n        if not candidate_masses:\n            # Fully malformed precursor: deterministic structure-only fallback; no popularity.\n            for proxy, smiles, mass in db.execute(\n                "SELECT proxy,MIN(canonical),MIN(mass) FROM structures "\n                "GROUP BY proxy ORDER BY proxy LIMIT ?",\n                (config["max_predictions"] * 4,),\n            ):\n                if proxy not in excluded:\n                    canonical[proxy] = smiles\n                    candidate_masses[proxy].add(mass)\n        scores = {}\n        for proxy, masses in candidate_masses.items():\n            support = []\n            for center, scale, guessed in hypotheses:\n                sigma = max(\n                    center * config["mass_score_ppm"] / 1e6, config["mass_score_floor_da"] * scale\n                )\n                residual = min(\n                    (abs(m - center) / sigma for m in masses if m is not None), default=float("inf")\n                )\n                support.append((1 / (1 + residual * residual)) * (0.5 if guessed else 1.0))\n            # Heavy-tail, best-observation + weak mean support. Never intersect candidates.\n            scores[proxy] = 0.8 * max(support, default=0) + 0.2 * float(\n                np.mean(support) if support else 0\n            )\n        ordered = sorted(scores, key=lambda p: (-scores[p], p))[: config["max_candidates"]]\n        pools[molecule] = {p: scores[p] for p in ordered}\n        intersection = set.intersection(*strict_sets) if strict_sets else set()\n        details[molecule] = {\n            "mass_union_size": len(union),\n            "pool_before_cap": len(scores),\n            "pool_size": len(ordered),\n            "mass_union": sorted(union),\n            "mass_intersection": sorted(intersection),\n            "intersection_empty_fallback": sorted(intersection or union),\n            "hypothesis_count": len(hypotheses),\n            "fallback_guessed": any(h[2] for h in hypotheses),\n        }\n    return pools, canonical, details\n\n\ndef entropy_similarity(left: tuple, right: tuple) -> float:\n    a, av = left\n    b, bv = right\n    if not len(a) or not len(b):\n        return 0.0\n    pos = np.searchsorted(b, a)\n    ids = np.flatnonzero(pos < len(b))\n    ids = ids[b[pos[ids]] == a[ids]]\n    p, q = av[ids] ** 2, bv[pos[ids]] ** 2\n    total = p + q\n    score = 0.5 * np.sum(total * np.log2(total) - p * np.log2(p) - q * np.log2(q))\n    return float(np.clip(score, 0, 1))\n\n\ndef reference_mapping(db: sqlite3.Connection, pool: set[str]) -> dict[str, str]:\n    result = {}\n    ordered = sorted(pool)\n    for i in range(0, len(ordered), 500):\n        subset = ordered[i : i + 500]\n        placeholders = ",".join("?" for _ in subset)\n        for smiles, proxy in db.execute(\n            f"SELECT smiles,proxy FROM structures WHERE proxy IN ({placeholders})", subset\n        ):\n            result[smiles] = proxy\n    return result\n\n\ndef retrieve(\n    train: Path,\n    queries: list[Query],\n    pools: dict,\n    mapping: dict,\n    config: dict,\n    validation: dict | None = None,\n) -> tuple[dict, dict]:\n    """One full bounded scan. Per-query max over reference spectra per normalized structure."""\n    start = time.monotonic()\n    reverse = collections.defaultdict(list)\n    for qi, query in enumerate(queries):\n        for proxy in pools[query.molecule]:\n            reverse[proxy].append(qi)\n    variants = (\n        ["cosine", "entropy"]\n        if validation is None\n        else ["cosine", "entropy", "structure_cosine", "structure_entropy"]\n    )\n    best = {name: [dict() for _ in queries] for name in variants}\n    counts = collections.Counter()\n    blocked_hashes = set(q.peak_digest for q in queries) if validation else set()\n    removed_structures = set(validation.get("all_proxy_keys", [])) if validation else set()\n    held_sources = validation.get("held_sources", {}) if validation else {}\n    self_target = validation.get("targets", {}) if validation else {}\n    if validation:\n        # Eligibility is independent of mass-pool recall. These diagnostic-only\n        # pairs never enter ranking unless the target is actually in the pool.\n        for qi, query in enumerate(queries):\n            target = self_target.get(query.molecule)\n            if target is not None and qi not in reverse[target]:\n                reverse[target].append(qi)\n    available_references = set()\n    pf = pq.ParquetFile(train, memory_map=True, pre_buffer=False)\n    for rg in range(pf.num_row_groups):\n        for batch in pf.iter_batches(\n            batch_size=config["arrow_batch_rows"],\n            row_groups=[rg],\n            columns=TRAIN_COLUMNS,\n            use_threads=False,\n        ):\n            counts["train_rows_scanned"] += len(batch)\n            smiles_values = batch.column("normalized_smiles").to_pylist()\n            selected = [i for i, smiles in enumerate(smiles_values) if smiles in mapping]\n            if not selected:\n                continue\n            small = batch.take(pa.array(selected, type=pa.int32()))\n            for row in small.to_pylist():\n                counts["candidate_reference_rows"] += 1\n                proxy = mapping[row["normalized_smiles"]]\n                if validation:\n                    digest = peak_hash(row["ms2_mzs"], row["ms2_normalized_intensities"])\n                    if digest in blocked_hashes:\n                        counts["exact_peak_rows_removed_all_labels"] += 1\n                        continue\n                spectrum = binned_peaks(\n                    row["ms2_mzs"], row["ms2_normalized_intensities"], config["bin_width_da"]\n                )\n                ref_ce = mean_ce(row["collision_energy_ev"])\n                ref_parts = adduct_parts(row["adduct"])\n                ref_pol = row["ionization_mode"]\n                ref_contradiction = bool(\n                    ref_parts\n                    and ref_pol in {"positive", "negative"}\n                    and ((ref_parts[1] > 0) != (ref_pol == "positive"))\n                )\n                for qi in reverse[proxy]:\n                    query = queries[qi]\n                    if validation and row["ingest_lib"] == held_sources.get(query.molecule):\n                        counts["held_source_pairs_removed"] += 1\n                        continue\n                    if (\n                        query.polarity in {"positive", "negative"}\n                        and ref_pol in {"positive", "negative"}\n                        and query.polarity != ref_pol\n                        and not query.contradiction\n                        and not ref_contradiction\n                    ):\n                        continue\n                    cosine = sparse_cosine(query.peaks, spectrum)\n                    counts["spectral_pairs"] += 1\n                    if (\n                        validation\n                        and cosine >= config["near_duplicate_cosine"]\n                        and query.precursor is not None\n                        and row["precursor_mz"] is not None\n                        and abs(query.precursor - row["precursor_mz"])\n                        <= config["near_duplicate_precursor_da"]\n                    ):\n                        counts["near_duplicate_pairs_removed_all_labels"] += 1\n                        continue\n                    if validation and self_target.get(query.molecule) == proxy:\n                        available_references.add(query.molecule)\n                    if proxy not in pools[query.molecule]:\n                        continue\n                    entropy = entropy_similarity(query.peaks, spectrum)\n                    ce_weight = 1.0\n                    if query.ce is not None and ref_ce is not None:\n                        ce_weight = 1 / (\n                            1 + abs(query.ce - ref_ce) / config["collision_energy_scale_ev"]\n                        )\n                    for name, score in (("cosine", cosine), ("entropy", entropy)):\n                        value = score * ce_weight\n                        best[name][qi][proxy] = max(best[name][qi].get(proxy, 0.0), value)\n                        if validation and proxy not in removed_structures:\n                            key = "structure_" + name\n                            best[key][qi][proxy] = max(best[key][qi].get(proxy, 0.0), value)\n        print(\n            f"retrieval row_group={rg + 1}/{pf.num_row_groups} "\n            f"rows={counts[\'train_rows_scanned\']} pairs={counts[\'spectral_pairs\']} "\n            f"seconds={time.monotonic() - start:.1f}",\n            flush=True,\n        )\n        pa.default_memory_pool().release_unused()\n    counts["wall_seconds"] = time.monotonic() - start\n    counts["peak_rss_kib"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    return best, {\n        "counts": dict(counts),\n        "reference_available_molecules": sorted(available_references),\n    }\n\n\ndef ranked_scores(scores: dict, config: dict) -> dict:\n    return {\n        p: (config["rrf_k"] + 1) / (config["rrf_k"] + rank)\n        for rank, p in enumerate(sorted(scores, key=lambda p: (-scores[p], p)), 1)\n        if scores[p] > 0\n    }\n\n\ndef rank_molecules(\n    queries: list[Query],\n    pools: dict,\n    per_query: list[dict],\n    config: dict,\n    fusion: str = "condition_mean",\n) -> dict:\n    groups = collections.defaultdict(list)\n    for qi, query in enumerate(queries):\n        groups[query.molecule].append(qi)\n    outputs = {}\n    for molecule, indices in groups.items():\n        conditions = collections.defaultdict(list)\n        for qi in indices:\n            conditions[queries[qi].condition].append(ranked_scores(per_query[qi], config))\n        # Max within acquisition condition; mean across distinct conditions, not row counts.\n        condition_scores = []\n        for lists in conditions.values():\n            aggregate = {}\n            for scores in lists:\n                for p, score in scores.items():\n                    aggregate[p] = max(aggregate.get(p, 0), score)\n            condition_scores.append(aggregate)\n        total = {}\n        for p, mass in pools[molecule].items():\n            evidence = [scores.get(p, 0.0) for scores in condition_scores]\n            spectral = (\n                max(evidence, default=0)\n                if fusion == "max"\n                else float(np.mean(evidence) if evidence else 0)\n            )\n            total[p] = config["mass_weight"] * mass + config["spectral_weight"] * spectral\n        outputs[molecule] = sorted(total, key=lambda p: (-total[p], p))\n    return outputs\n\n\ndef write_submission(\n    path: Path, ids: list[str], predictions: dict, canonical: dict, max_predictions: int = 25\n) -> dict:\n    if not isinstance(max_predictions, int) or not 1 <= max_predictions <= 25:\n        raise ValueError("max_predictions must be an integer between 1 and 25")\n    if len(ids) != len(set(ids)) or set(ids) != set(predictions):\n        raise ValueError("Submission IDs must be unique and exactly match runtime query IDs")\n    if not ids:\n        raise ValueError("Submission must contain at least one molecule")\n    rows, counts = [], []\n    # Validate every row before exclusively creating the final filename.\n    for molecule in ids:\n        output, seen = [], set()\n        for proxy in predictions[molecule]:\n            smiles = canonical[proxy]\n            norm = normalize_smiles(smiles)\n            if norm is None or norm[0] != proxy:\n                raise ValueError("Invalid or inconsistent normalized candidate")\n            if proxy not in seen:\n                output.append(smiles)\n                seen.add(proxy)\n            if len(output) == max_predictions:\n                break\n        if not output:\n            raise ValueError(f"Empty candidate list for {molecule}")\n        counts.append(len(output))\n        rows.append([molecule, ";".join(output)])\n    with path.open("x", newline="", encoding="utf-8") as stream:\n        writer = csv.writer(stream)\n        writer.writerow(["molecule_id", "smiles"])\n        writer.writerows(rows)\n    return {\n        "rows": len(ids),\n        "columns": ["molecule_id", "smiles"],\n        "min_candidates": min(counts),\n        "max_candidates": max(counts),\n        "invalid_candidates": 0,\n        "normalized_duplicate_candidates": 0,\n        "sha256": sha256_file(path),\n    }\n\n\ndef infer(train: Path, test: Path, catalog: Path, output: Path, config_path: Path) -> dict:\n    output.mkdir(parents=True, exist_ok=False)\n    started = time.monotonic()\n    config = json.loads(config_path.read_text())\n    queries, ids, query_report = read_queries(test, config)\n    db = open_catalog(catalog)\n    expected = db.execute("SELECT value FROM provenance WHERE key=\'train_sha256\'").fetchone()[0]\n    if sha256_file(train) != expected:\n        raise ValueError("Reference catalog and train SHA256 mismatch")\n    pools, canonical, candidate_details = make_candidates(queries, db, config)\n    mapping = reference_mapping(db, set().union(*(set(p) for p in pools.values())))\n    db.close()\n    best, retrieval = retrieve(train, queries, pools, mapping, config)\n    predictions = rank_molecules(queries, pools, best["cosine"], config)\n    submission = write_submission(\n        output / "submission.csv", ids, predictions, canonical, config["max_predictions"]\n    )\n    summary = {\n        "kind": "provided-test format/inference run, NOT unseen performance",\n        "versions": runtime_versions(),\n        "config_sha256": sha256_file(config_path),\n        "train_sha256": expected,\n        "test_sha256": sha256_file(test),\n        "catalog_sha256": sha256_file(catalog),\n        "queries": query_report,\n        "retrieval": retrieval,\n        "submission": submission,\n        "wall_seconds": time.monotonic() - started,\n        "peak_rss_kib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,\n        "pool_sizes": {q: candidate_details[q]["pool_size"] for q in ids},\n    }\n    write_json(output / "inference_manifest_PRIVATE.json", summary)\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--train", required=True, type=Path)\n    parser.add_argument("--test", required=True, type=Path)\n    parser.add_argument("--catalog", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument(\n        "--config", type=Path, default=Path(__file__).with_name("casmi_config_v1.json")\n    )\n    args = parser.parse_args()\n    result = infer(args.train, args.test, args.catalog, args.output, args.config)\n    print(json.dumps(result["submission"], indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'casmi_cpu_baseline_v3.py': '"""Current guarded production entrypoint: identical frozen ranks, less unused work.\n\nOnly unused entropy computation is omitted. Candidate sets, cosine scores,\ncondition aggregation, mass scores, ordering and SMILES output are unchanged.\nValidation continues through the complete v2 diagnostic path.\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport resource\nimport time\nfrom pathlib import Path\n\nfrom casmi_chemistry import runtime_versions, sha256_file, write_json\nfrom casmi_cpu_baseline import (\n    make_candidates,\n    open_catalog,\n    rank_molecules,\n    read_queries,\n    reference_mapping,\n    write_submission,\n)\nfrom casmi_retrieval_fast_v1 import retrieve\nfrom catalog_contract_v2 import verify_complete_catalog\n\n\ndef infer(train: Path, test: Path, catalog: Path, output: Path, config_path: Path) -> dict:\n    started = time.monotonic()\n    catalog_manifest = verify_complete_catalog(catalog)\n    output.mkdir(parents=True, exist_ok=False)\n    config = json.loads(config_path.read_text())\n    queries, ids, query_report = read_queries(test, config)\n    db = open_catalog(catalog)\n    expected = db.execute("SELECT value FROM provenance WHERE key=\'train_sha256\'").fetchone()[0]\n    if sha256_file(train) != expected:\n        raise ValueError("Reference catalog and train SHA256 mismatch")\n    pools, canonical, candidate_details = make_candidates(queries, db, config)\n    mapping = reference_mapping(db, set().union(*(set(p) for p in pools.values())))\n    db.close()\n    best, retrieval = retrieve(train, queries, pools, mapping, config, metrics=("cosine",))\n    predictions = rank_molecules(queries, pools, best["cosine"], config)\n    submission = write_submission(\n        output / "submission.csv", ids, predictions, canonical, config["max_predictions"]\n    )\n    summary = {\n        "kind": "provided-test format/inference run, NOT unseen performance",\n        "optimization": "omit unused entropy only; frozen model unchanged",\n        "versions": runtime_versions(),\n        "config_sha256": sha256_file(config_path),\n        "train_sha256": expected,\n        "test_sha256": sha256_file(test),\n        "catalog_sha256": catalog_manifest["catalog_sha256"],\n        "queries": query_report,\n        "retrieval": retrieval,\n        "submission": submission,\n        "wall_seconds": time.monotonic() - started,\n        "peak_rss_kib": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss,\n        "pool_sizes": {q: candidate_details[q]["pool_size"] for q in ids},\n    }\n    write_json(output / "inference_manifest_PRIVATE.json", summary)\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--train", required=True, type=Path)\n    parser.add_argument("--test", required=True, type=Path)\n    parser.add_argument("--catalog", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument(\n        "--config", type=Path, default=Path(__file__).with_name("casmi_config_v1.json")\n    )\n    args = parser.parse_args()\n    result = infer(args.train, args.test, args.catalog, args.output, args.config)\n    print(json.dumps(result["submission"], indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'casmi_offline_bootstrap.py': '"""CPython 3.13/Linux x86_64 local-only CASMI dependency bootstrap.\n\nCopy this code into the notebook before importing NumPy, RDKit, Arrow, or PIL.\nCall install_offline_runtime(WHEELHOUSE, writable_working_directory).\nNo pip, package index, network access, or existing uv installation is used.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib\nimport importlib.metadata\nimport json\nimport os\nimport platform\nimport subprocess\nimport sys\nimport sysconfig\nimport tempfile\nimport zipfile\nfrom pathlib import Path\n\nWHEEL_SHA256 = dict(\n    [\n        (\n            "rdkit-2026.3.3-cp313-cp313-manylinux_2_28_x86_64.whl",\n            "9a21f96d92c434c32a4c0d1cc6ebde5f0e57ddbbf24cbfbfd085a3a80e1ceab3",\n        ),\n        (\n            "numpy-2.3.5-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl",\n            "11e06aa0af8c0f05104d56450d6093ee639e15f24ecf62d417329d06e522e017",\n        ),\n        (\n            "pyarrow-25.0.1-cp313-cp313-manylinux_2_28_x86_64.whl",\n            "0befcf816e45a1af33ac775a9970b749e4868a230c7372f0ae5e932bee27039f",\n        ),\n        (\n            "pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl",\n            "0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3",\n        ),\n        (\n            "uv-0.12.19-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",\n            "a63d18a0aa38ee9f21a5406afbbaeb41303bcd954be9d6b7c1b95ac275e53958",\n        ),\n    ]\n)\nRUNTIME_PINS = {"rdkit": "2026.3.3", "numpy": "2.3.5", "pyarrow": "25.0.1", "pillow": "12.3.0"}\n\n\ndef _requirements_lock():\n    lines = []\n    for package, version in RUNTIME_PINS.items():\n        filename = next(name for name in WHEEL_SHA256 if name.startswith(package + "-"))\n        lines.append(f"{package}=={version} --hash=sha256:{WHEEL_SHA256[filename]}")\n    return "\\n".join(lines) + "\\n"\n\n\nREQUIREMENTS_LOCK = _requirements_lock()\n\n\ndef install_offline_runtime(wheelhouse, working_directory):\n    """Verify pinned wheels; install into a new private target; prepend sys.path."""\n    if sys.implementation.name != "cpython" or sys.version_info[:2] != (3, 13):\n        raise RuntimeError(\n            "Requires CPython 3.13 (cp313); obtain matching wheels for other Python versions."\n        )\n    # cp313 and free-threaded cp313t are different binary ABIs.\n    soabi = sysconfig.get_config_var("SOABI") or ""\n    if sysconfig.get_config_var("Py_GIL_DISABLED") or not soabi.startswith("cpython-313-"):\n        raise RuntimeError(f"Requires standard CPython 3.13 (cp313), not cp313t; detected {soabi}.")\n    if platform.system() != "Linux" or platform.machine() != "x86_64":\n        raise RuntimeError("This asset bundle requires Linux x86_64.")\n    libc_name, libc_version = platform.libc_ver()\n    try:\n        libc_tuple = tuple(int(x) for x in libc_version.split(".")[:2])\n    except ValueError:\n        libc_tuple = ()\n    if libc_name != "glibc" or libc_tuple < (2, 28):\n        raise RuntimeError(\n            f"This asset bundle requires glibc >= 2.28; detected {libc_name} {libc_version}."\n        )\n    already_loaded = [name for name in ("numpy", "rdkit", "pyarrow", "PIL") if name in sys.modules]\n    if already_loaded:\n        raise RuntimeError(f"Restart the kernel and bootstrap before importing: {already_loaded}")\n    wheelhouse = Path(wheelhouse).resolve(strict=True)\n    if not wheelhouse.is_dir():\n        raise RuntimeError(f"Wheelhouse is not a directory: {wheelhouse}")\n    for filename, expected in WHEEL_SHA256.items():\n        path = wheelhouse / filename\n        if not path.is_file():\n            raise RuntimeError(f"Missing offline wheel: {filename}")\n        with path.open("rb") as stream:\n            actual = hashlib.file_digest(stream, "sha256").hexdigest()\n        if actual != expected:\n            raise RuntimeError(f"Offline wheel SHA-256 mismatch: {filename}")\n    work = Path(working_directory).resolve(strict=True)\n    if not work.is_dir():\n        raise RuntimeError(f"Working directory is not a directory: {work}")\n    private = Path(tempfile.mkdtemp(prefix="casmi-offline-", dir=work))\n    target = private / "site-packages"\n    lock = private / "requirements-runtime.txt"\n    lock.write_text(REQUIREMENTS_LOCK)\n    uv_filename = "uv-0.12.19-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl"\n    executable = private / "uv"\n    with zipfile.ZipFile(wheelhouse / uv_filename) as archive:\n        executable.write_bytes(archive.read("uv-0.12.19.data/scripts/uv"))\n    executable.chmod(0o700)\n    # Keep unrelated UV_* defaults from affecting this explicitly offline installation.\n    env = {key: value for key, value in os.environ.items() if not key.startswith("UV_")}\n    env.update({"UV_OFFLINE": "1", "UV_NO_CACHE": "1", "UV_PYTHON_DOWNLOADS": "never"})\n    command = [\n        str(executable),\n        "pip",\n        "install",\n        "--python",\n        sys.executable,\n        "--target",\n        str(target),\n        "--offline",\n        "--no-index",\n        "--no-cache",\n        "--no-config",\n        "--no-python-downloads",\n        "--no-progress",\n        "--only-binary",\n        ":all:",\n        "--require-hashes",\n        "--find-links",\n        str(wheelhouse),\n        "-r",\n        str(lock),\n    ]\n    run = subprocess.run(command, env=env, text=True, capture_output=True)\n    (private / "install.stdout.log").write_text(run.stdout)\n    (private / "install.stderr.log").write_text(run.stderr)\n    if run.returncode:\n        raise RuntimeError(f"Local-only uv install failed ({run.returncode}):\\n{run.stderr}")\n    installed = {\n        dist.metadata["Name"].lower().replace("_", "-"): dist.version\n        for dist in importlib.metadata.distributions(path=[str(target)])\n    }\n    if installed != RUNTIME_PINS:\n        raise RuntimeError(\n            f"Installed dependency closure differs from pinned versions: {installed}"\n        )\n    sys.path.insert(0, str(target))\n    importlib.invalidate_caches()\n    report = {\n        "status": "installed",\n        "python": platform.python_version(),\n        "platform": platform.platform(),\n        "libc": [libc_name, libc_version],\n        "target": str(target),\n        "versions": installed,\n        "verified_wheels": dict(WHEEL_SHA256),\n        "installer": "uv 0.12.19",\n        "network": "disabled by uv --offline --no-index",\n        "kaggle_runtime_checked": False,\n    }\n    (private / "install-report.json").write_text(json.dumps(report, indent=2) + "\\n")\n    return report\n', 'casmi_retrieval_fast_v1.py': '"""Additive retrieval variant; chemistry and ranking contracts are unchanged.\n\nThe default retains all baseline diagnostics. Production cosine-only mode skips\nonly the unused entropy calculation, and cannot be used for validation.\nDerived from casmi_cpu_baseline.retrieve, preserved as a historical baseline.\n"""\n\nfrom __future__ import annotations\n\nimport collections\nimport resource\nimport time\nfrom pathlib import Path\n\nimport pyarrow as pa\nimport pyarrow.parquet as pq\nfrom casmi_chemistry import adduct_parts, binned_peaks, peak_hash, sparse_cosine\nfrom casmi_cpu_baseline import TRAIN_COLUMNS, Query, entropy_similarity, mean_ce\n\n\ndef retrieve(\n    train: Path,\n    queries: list[Query],\n    pools: dict,\n    mapping: dict,\n    config: dict,\n    validation: dict | None = None,\n    metrics: tuple[str, ...] = ("cosine", "entropy"),\n) -> tuple[dict, dict]:\n    """One full bounded scan. Per-query max over reference spectra per normalized structure."""\n    if metrics not in {("cosine",), ("cosine", "entropy")}:\n        raise ValueError("metrics must be cosine-only or cosine+entropy")\n    if validation is not None and metrics != ("cosine", "entropy"):\n        raise ValueError("Validation requires both metrics; no diagnostic bypass")\n    start = time.monotonic()\n    reverse = collections.defaultdict(list)\n    for qi, query in enumerate(queries):\n        for proxy in pools[query.molecule]:\n            reverse[proxy].append(qi)\n    variants = (\n        list(metrics)\n        if validation is None\n        else ["cosine", "entropy", "structure_cosine", "structure_entropy"]\n    )\n    best = {name: [{} for _ in queries] for name in variants}\n    counts = collections.Counter()\n    blocked_hashes = {q.peak_digest for q in queries} if validation else set()\n    removed_structures = set(validation.get("all_proxy_keys", [])) if validation else set()\n    held_sources = validation.get("held_sources", {}) if validation else {}\n    self_target = validation.get("targets", {}) if validation else {}\n    if validation:\n        # Eligibility is independent of mass-pool recall. These diagnostic-only\n        # pairs never enter ranking unless the target is actually in the pool.\n        for qi, query in enumerate(queries):\n            target = self_target.get(query.molecule)\n            if target is not None and qi not in reverse[target]:\n                reverse[target].append(qi)\n    available_references = set()\n    pf = pq.ParquetFile(train, memory_map=True, pre_buffer=False)\n    for rg in range(pf.num_row_groups):\n        for batch in pf.iter_batches(\n            batch_size=config["arrow_batch_rows"],\n            row_groups=[rg],\n            columns=TRAIN_COLUMNS,\n            use_threads=False,\n        ):\n            counts["train_rows_scanned"] += len(batch)\n            smiles_values = batch.column("normalized_smiles").to_pylist()\n            selected = [i for i, smiles in enumerate(smiles_values) if smiles in mapping]\n            if not selected:\n                continue\n            small = batch.take(pa.array(selected, type=pa.int32()))\n            for row in small.to_pylist():\n                counts["candidate_reference_rows"] += 1\n                proxy = mapping[row["normalized_smiles"]]\n                if validation:\n                    digest = peak_hash(row["ms2_mzs"], row["ms2_normalized_intensities"])\n                    if digest in blocked_hashes:\n                        counts["exact_peak_rows_removed_all_labels"] += 1\n                        continue\n                spectrum = binned_peaks(\n                    row["ms2_mzs"],\n                    row["ms2_normalized_intensities"],\n                    config["bin_width_da"],\n                )\n                ref_ce = mean_ce(row["collision_energy_ev"])\n                ref_parts = adduct_parts(row["adduct"])\n                ref_pol = row["ionization_mode"]\n                ref_contradiction = bool(\n                    ref_parts\n                    and ref_pol in {"positive", "negative"}\n                    and ((ref_parts[1] > 0) != (ref_pol == "positive"))\n                )\n                for qi in reverse[proxy]:\n                    query = queries[qi]\n                    if validation and row["ingest_lib"] == held_sources.get(query.molecule):\n                        counts["held_source_pairs_removed"] += 1\n                        continue\n                    if (\n                        query.polarity in {"positive", "negative"}\n                        and ref_pol in {"positive", "negative"}\n                        and query.polarity != ref_pol\n                        and not query.contradiction\n                        and not ref_contradiction\n                    ):\n                        continue\n                    cosine = sparse_cosine(query.peaks, spectrum)\n                    counts["spectral_pairs"] += 1\n                    if (\n                        validation\n                        and cosine >= config["near_duplicate_cosine"]\n                        and query.precursor is not None\n                        and row["precursor_mz"] is not None\n                        and abs(query.precursor - row["precursor_mz"])\n                        <= config["near_duplicate_precursor_da"]\n                    ):\n                        counts["near_duplicate_pairs_removed_all_labels"] += 1\n                        continue\n                    if validation and self_target.get(query.molecule) == proxy:\n                        available_references.add(query.molecule)\n                    if proxy not in pools[query.molecule]:\n                        continue\n                    entropy = (\n                        entropy_similarity(query.peaks, spectrum) if "entropy" in metrics else None\n                    )\n                    ce_weight = 1.0\n                    if query.ce is not None and ref_ce is not None:\n                        ce_weight = 1 / (\n                            1 + abs(query.ce - ref_ce) / config["collision_energy_scale_ev"]\n                        )\n                    for name, score in (("cosine", cosine), ("entropy", entropy)):\n                        if name not in metrics:\n                            continue\n                        value = score * ce_weight\n                        best[name][qi][proxy] = max(best[name][qi].get(proxy, 0.0), value)\n                        if validation and proxy not in removed_structures:\n                            key = "structure_" + name\n                            best[key][qi][proxy] = max(best[key][qi].get(proxy, 0.0), value)\n        print(\n            f"retrieval row_group={rg + 1}/{pf.num_row_groups} "\n            f"rows={counts[\'train_rows_scanned\']} pairs={counts[\'spectral_pairs\']} "\n            f"seconds={time.monotonic() - start:.1f}",\n            flush=True,\n        )\n        pa.default_memory_pool().release_unused()\n    counts["wall_seconds"] = time.monotonic() - start\n    counts["peak_rss_kib"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss\n    return best, {\n        "counts": dict(counts),\n        "reference_available_molecules": sorted(available_references),\n    }\n', 'catalog_contract_v2.py': '"""Reject incomplete, mismatched or smoke-only reference assets before prediction."""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\n\nfrom casmi_chemistry import runtime_versions, sha256_file\nfrom casmi_cpu_baseline import open_catalog\n\nNORMALIZATION_CONTRACT = "RDKit-default-TautomerEnumerator-Canonicalize-InChIKey14-v1"\n\n\ndef verify_complete_catalog(catalog: Path, *, allow_limited_smoke: bool = False) -> dict:\n    manifest_path = catalog.with_name("manifest.json")\n    if not manifest_path.is_file():\n        raise ValueError("Incomplete/unverified catalog: adjacent final manifest.json required")\n    manifest = json.loads(manifest_path.read_text())\n    if manifest.get("limited_smoke") and not allow_limited_smoke:\n        raise ValueError("Limited smoke catalog cannot be used for production inference")\n    if manifest.get("versions", {}).get("rdkit") != runtime_versions()["rdkit"]:\n        raise ValueError("Catalog normalization RDKit version mismatch")\n    if manifest.get("catalog_sha256") != sha256_file(catalog):\n        raise ValueError("Catalog final-manifest SHA256 mismatch")\n    if manifest.get("normalization_contract") != NORMALIZATION_CONTRACT:\n        raise ValueError("Catalog normalization contract mismatch")\n    expected_valid = manifest.get("valid_variants")\n    expected_invalid = manifest.get("invalid_variants")\n    expected_unique = manifest.get("unique_variants")\n    if not all(\n        isinstance(value, int) and value >= 0\n        for value in (expected_valid, expected_invalid, expected_unique)\n    ):\n        raise ValueError("Catalog final manifest lacks complete row accounting")\n    if expected_valid + expected_invalid != expected_unique or expected_valid == 0:\n        raise ValueError("Incomplete catalog normalized-variant accounting")\n    db = open_catalog(catalog)\n    try:\n        actual = db.execute("SELECT COUNT(*) FROM structures").fetchone()[0]\n        provenance = dict(db.execute("SELECT key,value FROM provenance"))\n    finally:\n        db.close()\n    if actual != expected_valid or provenance.get("train_sha256") != manifest.get("train_sha256"):\n        raise ValueError("Catalog row count/train provenance differs from final manifest")\n    return manifest\n', 'official_wheels_cp313.json': '[\n  {\n    "filename": "rdkit-2026.3.3-cp313-cp313-manylinux_2_28_x86_64.whl",\n    "url": "https://files.pythonhosted.org/packages/d3/1a/a8b64f5304c19181fb4a0b859b7b4bd07d15b4e1379deb024d72cb012b9e/rdkit-2026.3.3-cp313-cp313-manylinux_2_28_x86_64.whl",\n    "sha256": "9a21f96d92c434c32a4c0d1cc6ebde5f0e57ddbbf24cbfbfd085a3a80e1ceab3",\n    "size": 37187145,\n    "package": "rdkit",\n    "version": "2026.3.3"\n  },\n  {\n    "filename": "numpy-2.3.5-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl",\n    "url": "https://files.pythonhosted.org/packages/f5/10/ca162f45a102738958dcec8023062dad0cbc17d1ab99d68c4e4a6c45fb2b/numpy-2.3.5-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl",\n    "sha256": "11e06aa0af8c0f05104d56450d6093ee639e15f24ecf62d417329d06e522e017",\n    "size": 16597430,\n    "package": "numpy",\n    "version": "2.3.5"\n  },\n  {\n    "filename": "pyarrow-25.0.1-cp313-cp313-manylinux_2_28_x86_64.whl",\n    "url": "https://files.pythonhosted.org/packages/98/d6/33a411115b61dbfc16ad6ad73e71730f6fea654ee3667673bc53ab0e2fe7/pyarrow-25.0.1-cp313-cp313-manylinux_2_28_x86_64.whl",\n    "sha256": "0befcf816e45a1af33ac775a9970b749e4868a230c7372f0ae5e932bee27039f",\n    "size": 50104452,\n    "package": "pyarrow",\n    "version": "25.0.1"\n  },\n  {\n    "filename": "pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl",\n    "url": "https://files.pythonhosted.org/packages/f7/62/de5bdd77d935331f4f802edc11e4d82950f642caad6cb2f949837b8560e2/pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl",\n    "sha256": "0847a763afefb695bc912d7c131e7e0632d4edc1d8698f58ddabec8e46b8b6d3",\n    "size": 6937129,\n    "package": "pillow",\n    "version": "12.3.0"\n  },\n  {\n    "filename": "uv-0.12.19-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",\n    "url": "https://files.pythonhosted.org/packages/76/71/b47cec536d8ee7b09017c1d9db211dfc2e7ce5c87d0482918d8b3411ec48/uv-0.12.19-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl",\n    "sha256": "a63d18a0aa38ee9f21a5406afbbaeb41303bcd954be9d6b7c1b95ac275e53958",\n    "size": 20478749,\n    "package": "uv",\n    "version": "0.12.19"\n  }\n]\n', 'prepare_kaggle_assets_v1.py': '"""Prepare frozen CASMI assets on Kaggle; no Kaggle API or test input is used."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport os\nimport shutil\nimport subprocess\nimport sys\nimport tempfile\nimport time\nimport urllib.parse\nimport urllib.request\nimport zipfile\nfrom pathlib import Path, PurePosixPath\n\n\ndef sha256(path):\n    with Path(path).open("rb") as stream:\n        return hashlib.file_digest(stream, "sha256").hexdigest()\n\n\ndef write_json(path, value):\n    with Path(path).open("x", encoding="utf-8") as stream:\n        json.dump(value, stream, indent=2)\n        stream.write("\\n")\n\n\ndef official_url(url):\n    parsed = urllib.parse.urlsplit(url)\n    if parsed.scheme != "https" or parsed.hostname != "files.pythonhosted.org":\n        raise ValueError("Wheel URL must use the official files.pythonhosted.org HTTPS host")\n    if parsed.username or parsed.password or parsed.port not in (None, 443):\n        raise ValueError("Unexpected wheel URL authority")\n\n\ndef fetch_wheel(record, wheelhouse, *, opener=urllib.request.urlopen):\n    """Stream a pinned official wheel and publish its filename only after verification."""\n    name = record["filename"]\n    if Path(name).name != name or not name.endswith(".whl"):\n        raise ValueError("Invalid wheel filename")\n    official_url(record["url"])\n    final = Path(wheelhouse) / name\n    if final.exists():\n        if final.stat().st_size == record["size"] and sha256(final) == record["sha256"]:\n            return final\n        raise FileExistsError(f"Existing wheel differs from the locked wheel: {name}")\n    partial = final.with_suffix(final.suffix + ".partial")\n    digest = hashlib.sha256()\n    size = 0\n    request = urllib.request.Request(record["url"], headers={"User-Agent": "CASMI-assets/1"})\n    with opener(request, timeout=120) as response, partial.open("xb") as output:\n        official_url(response.geturl())\n        while chunk := response.read(1024 * 1024):\n            size += len(chunk)\n            if size > record["size"]:\n                raise ValueError(f"Wheel exceeds locked size: {name}")\n            digest.update(chunk)\n            output.write(chunk)\n    if size != record["size"] or digest.hexdigest() != record["sha256"]:\n        raise ValueError(f"Wheel size/SHA256 mismatch: {name}; partial file retained")\n    # Exclusive final creation also protects against a concurrent writer.\n    with partial.open("rb") as source, final.open("xb") as destination:\n        shutil.copyfileobj(source, destination)\n    partial.unlink()  # Only this function\'s verified temporary download is removed.\n    return final\n\n\ndef preserve_licenses(wheelhouse, destination):\n    for wheel in sorted(Path(wheelhouse).glob("*.whl")):\n        with zipfile.ZipFile(wheel) as archive:\n            for name in archive.namelist():\n                member = PurePosixPath(name)\n                if member.is_absolute() or ".." in member.parts:\n                    raise ValueError("Unsafe wheel member")\n                if not any(\n                    token in member.name.lower() for token in ("license", "notice", "copying")\n                ):\n                    continue\n                if name.endswith("/"):\n                    continue\n                target = Path(destination) / wheel.stem / member\n                target.parent.mkdir(parents=True, exist_ok=True)\n                with target.open("xb") as stream:\n                    stream.write(archive.read(name))\n\n\ndef prepare(train, output, source_dir, reuse_catalog=None):\n    started = time.monotonic()\n    train = Path(train).resolve(strict=True)\n    source_dir = Path(source_dir).resolve(strict=True)\n    output = Path(output)\n    records = json.loads((source_dir / "official_wheels_cp313.json").read_text())\n    from casmi_offline_bootstrap import WHEEL_SHA256, install_offline_runtime\n\n    if {row["filename"]: row["sha256"] for row in records} != WHEEL_SHA256:\n        raise ValueError("Download manifest and frozen bootstrap hashes differ")\n    output.mkdir(parents=True, exist_ok=False)\n    wheelhouse = output / "wheelhouse"\n    wheelhouse.mkdir()\n    for record in records:\n        print("Official wheel:", record["filename"], flush=True)\n        fetch_wheel(record, wheelhouse)\n    write_json(output / "official_wheels_cp313.json", records)\n    preserve_licenses(wheelhouse, output / "licenses")\n    runtime_work = Path(tempfile.mkdtemp(prefix="casmi-preparation-runtime-"))\n    runtime = install_offline_runtime(wheelhouse, runtime_work)\n    os.environ["PYTHONPATH"] = runtime["target"] + os.pathsep + str(source_dir)\n    from casmi_chemistry import runtime_versions\n    from catalog_contract_v2 import verify_complete_catalog\n\n    train_hash = sha256(train)\n    catalog_dir = output / "private_catalog"\n    if reuse_catalog is not None:\n        reuse_catalog = Path(reuse_catalog).resolve(strict=True)\n        manifest = verify_complete_catalog(reuse_catalog)\n        if manifest["train_sha256"] != train_hash:\n            raise ValueError("Attached catalog and current train SHA256 differ")\n        catalog_dir.mkdir()\n        for filename in ("catalog.sqlite", "manifest.json", "build_contract.json"):\n            source = reuse_catalog.with_name(filename)\n            with source.open("rb") as incoming, (catalog_dir / filename).open("xb") as target:\n                shutil.copyfileobj(incoming, target)\n        mode = "verified-attached-catalog-copy"\n    else:\n        print("Building catalog from competition train; two CPU workers", flush=True)\n        command = [\n            sys.executable,\n            "-S",\n            str(source_dir / "build_reference_catalog_v2.py"),\n            "--train",\n            str(train),\n            "--output",\n            str(catalog_dir),\n            "--workers",\n            "2",\n        ]\n        subprocess.run(command, check=True)\n        mode = "cold-build-from-train"\n    manifest = verify_complete_catalog(catalog_dir / "catalog.sqlite")\n    if manifest["train_sha256"] != train_hash:\n        raise ValueError("Prepared catalog and current train SHA256 differ")\n    files = {\n        path.relative_to(output).as_posix(): {"bytes": path.stat().st_size, "sha256": sha256(path)}\n        for path in sorted(output.rglob("*"))\n        if path.is_file()\n    }\n    result = {\n        "complete": True,\n        "kind": "private CASMI preparation output",\n        "versions": runtime_versions(),\n        "train_sha256": train_hash,\n        "catalog_sha256": manifest["catalog_sha256"],\n        "catalog_mode": mode,\n        "normalization_contract": manifest["normalization_contract"],\n        "test_data_read": False,\n        "prediction_or_scoring_run": False,\n        "source_sha256": {\n            path.name: sha256(path) for path in sorted(source_dir.iterdir()) if path.is_file()\n        },\n        "files": files,\n        "artifact_bytes": sum(item["bytes"] for item in files.values()),\n        "wall_seconds": time.monotonic() - started,\n    }\n    write_json(output / "asset_manifest.json", result)\n    print("PREPARATION COMPLETE", output, flush=True)\n    print("Save this completed Notebook version privately; attach its Output to inference.")\n    print("No submission.csv is created in preparation.")\n    return result\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--train", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--source-dir", required=True, type=Path)\n    parser.add_argument("--reuse-catalog", type=Path)\n    args = parser.parse_args()\n    os.environ["OMP_NUM_THREADS"] = "1"\n    os.environ["OPENBLAS_NUM_THREADS"] = "1"\n    if hasattr(os, "nice"):\n        os.nice(10)\n    prepare(args.train, args.output, args.source_dir, args.reuse_catalog)\n\n\nif __name__ == "__main__":\n    main()\n'}

for filename, contents in EMBEDDED_SOURCE.items():
    with (source_dir / filename).open("x", encoding="utf-8") as stream:
        stream.write(contents)
output = work_root / "casmi_prepared_assets_v1"
if output.exists():
    raise FileExistsError("既存の準備出力は上書きしません。fresh sessionで実行してください。")
command = [sys.executable, "-S", str(source_dir / "prepare_kaggle_assets_v1.py"),
           "--train", str(train_paths[0]), "--output", str(output),
           "--source-dir", str(source_dir)]
if catalog_paths:
    command += ["--reuse-catalog", str(catalog_paths[0])]
subprocess.run(command, check=True)
manifest = json.loads((output / "asset_manifest.json").read_text())
assert manifest["complete"]
print("準備完了:", output)
print("保存するasset bytes:", manifest["artifact_bytes"])
print("catalog:", manifest["catalog_mode"])
print("この成功した実行のOutputを、提出Notebookの入力に追加してください。")


Python: 3.13.15
Official wheel: rdkit-2026.3.3-cp313-cp313-manylinux_2_28_x86_64.whl
Official wheel: numpy-2.3.5-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Official wheel: pyarrow-25.0.1-cp313-cp313-manylinux_2_28_x86_64.whl
Official wheel: pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Official wheel: uv-0.12.19-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
Building catalog from competition train; two CPU workers
catalog input_rows=2539608 variants=277566 reused=0
normalized=1552/277566 pending_chunks=4 seconds=34.0
normalized=1600/277566 pending_chunks=4 seconds=49.5
normalized=1856/277566 pending_chunks=4 seconds=65.2
normalized=2288/277566 pending_chunks=4 seconds=80.4
normalized=2768/277566 pending_chunks=4 seconds=95.6
normalized=4304/277566 pending_chunks=4 seconds=111.4
normalized=4864/277566 pending_chunks=3 seconds=126.6
normalized=5168/277566 pending_chunks=3 seconds=141.7
normalized=5200/277566 pending_chunks=4 seconds=1

## 停止した場合

- ダウンロード不可：この準備NotebookだけInternet ONにし、fresh sessionで実行します。
  学習データを外部送信する処理はありません。アクセス先は固定の公式wheel配布URLだけです
- Python/ABIが違う：この版はstandard CPython 3.13 / Linux x86_64 / glibc 2.28以上が必要です
- 既存catalogとtrainが不一致：catalog入力を外して現在のtrainから新規構築します
- catalog構築が停止：完成manifestがない出力は提出に使えません。進捗と最後のエラーを確認してください
- 保存済みOutputが空：実行完了したversionを使ってください。Quick Saveのみのversionは避けます

公式のNotebook入力・保存動作の参考：
[Kaggle kernel metadata](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels_metadata.md)、
[Kaggle kernels](https://github.com/Kaggle/kaggle-cli/blob/main/docs/kernels.md)。
このコードからKaggle APIへのアクセス、Notebook公開、競技提出は行いません。
